# ANALYSIS OF DATA

# IMPORT LIBRARIES

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Enable IterativeImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer


# pip install optuna holidays scikit-learn

import numpy as np
import pandas as pd
import holidays
import optuna

from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit

# IMPORT DATA

In [ ]:
df1 = pd.read_excel("data/inputs/Principle_Data_Scientist_Tech_Assessment.xlsx", sheet_name = "daily records")

In [ ]:
df1.head()

# VALIDATION

In [ ]:
###check if a date is missing
###check if weekend is 0 or 1
###check if holiday is 0 or 1
###channel_mix_index not less than 0 and not more than 100
### media mention cannot be less than 0
### row_id int missing 

def validate_data(df):
    errors = {}

    # Ensure date column is datetime
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # -----------------------------
    # Missing date
    # -----------------------------
    errors["missing_date"] = df[df["date"].isna()]

    # -----------------------------
    # Date not in chronological order
    # -----------------------------
    errors["date_not_in_chronology"] = df[
        df["date"].diff().dt.days < 0
    ]

    # -----------------------------
    # row_id missing or not integer
    # -----------------------------
    errors["invalid_row_id"] = df[
        df["row_id"].isna() |
        (~df["row_id"].apply(
            lambda x: pd.isna(x) or float(x).is_integer()
        ))
    ]

    # -----------------------------
    # row_id not in chronological order
    # -----------------------------
    errors["row_id_not_in_order"] = df[
        df["row_id"].diff() < 0
    ]

    # -----------------------------
    # is_weekend must be 0 or 1
    # -----------------------------
    errors["invalid_is_weekend"] = df[
        ~df["is_weekend"].isin([0, 1])
    ]

    # -----------------------------
    # bank_holiday_flag must be 0 or 1
    # -----------------------------
    errors["invalid_bank_holiday_flag"] = df[
        ~df["bank_holiday_flag"].isin([0, 1])
    ]

    # -----------------------------
    # channel_mix_index between 0 and 100
    # NaN is allowed
    # -----------------------------
    errors["invalid_channel_mix_index"] = df[
        (~df["channel_mix_index"].between(0, 100)) &
        (df["channel_mix_index"].notna())
    ]

    # -----------------------------
    # media_mentions cannot be negative
    # -----------------------------
    errors["invalid_media_mentions"] = df[
        (df["media_mentions"] < 0) |
        (df["media_mentions"].isna())
    ]

    return errors

In [ ]:
validation_results = validate_data(df1)

for check_name, invalid_rows in validation_results.items():
    print(f"\n{check_name}: {len(invalid_rows)} invalid rows")

    if not invalid_rows.empty:
        print(invalid_rows)

# Missing DATA POINTS

In [ ]:
# Count NaN values in each column
nan_counts = df1.isna().sum()

print(nan_counts)

# Imputation

In [ ]:
def recover_missing_complaints(df):
    
    complaints = df["complaints"].copy()

    for i in range(3, len(df) - 3):

        # Skip if complaints already exists
        if pd.notna(complaints.iloc[i]):
            continue

        # Need centered mean available
        mean_val = df.loc[i, "centered_7d_mean"]

        if pd.isna(mean_val):
            continue

        # Get 7-day window
        window = complaints.iloc[i-3:i+4]

        # Recover only if exactly one missing value
        if window.isna().sum() == 1:

            known_sum = window.sum(skipna=True)

            missing_value = (mean_val * 7) - known_sum

            complaints.iloc[i] = missing_value

    df["recovered_complaints"] = complaints

    return df

In [ ]:
df1 = recover_missing_complaints(df1)

print(df1[["complaints", "recovered_complaints"]])

In [ ]:
print(
    df1[df1["complaints"].isna()][
        ["date", "complaints", "recovered_complaints", "centered_7d_mean"]
    ]
)

In [ ]:
# Count NaN values in each column
nan_counts = df1.isna().sum()

print(nan_counts)

In [ ]:
def impute_missing_values(df, seed=42):

    impute_cols = [
        "staffing_level_fte",
        "channel_mix_index",
        "backlog_days"
    ]

    predictor_cols = impute_cols + [
        "recovered_complaints",
        "media_mentions"
    ]

    # Store rows that were originally missing
    missing_mask = df[impute_cols].isna()

    # Copy data for imputation
    temp_df = df[predictor_cols].copy()

    # Create imputer
    imputer = IterativeImputer(
        random_state=seed,
        max_iter=50
    )

    # Impute
    imputed_array = imputer.fit_transform(temp_df)

    imputed_df = pd.DataFrame(
        imputed_array,
        columns=predictor_cols,
        index=df.index
    )

    # Replace only imputed columns
    for col in impute_cols:
        df[col] = imputed_df[col]

    # -----------------------------------
    # Show previously missing rows
    # -----------------------------------
    for col in impute_cols:

        missing_rows = missing_mask[col]

        if missing_rows.sum() > 0:

            print(f"\nImputed values for: {col}")

            print(
                df.loc[
                    missing_rows,
                    ["date", "row_id", col]
                ]
            )

    return df

In [ ]:
df1_imputed = impute_missing_values(df1)

## CLEANING AND VISUALIZATION

In [ ]:
def plot_time_series(df, date_col, columns, figsize=(12, 6)):
    """
    Plot multiple time series on a single y-axis.

    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataframe

    date_col : str
        Name of date column

    columns : list
        List of columns to plot

    figsize : tuple
        Figure size
    """

    plt.figure(figsize=figsize)

    # Plot each column
    for col in columns:
        plt.plot(df[date_col], df[col], label=col)

    # Labels and title
    plt.xlabel(date_col)
    plt.ylabel("Value")
    plt.title("Time Series Plot")

    # Legend
    plt.legend()

    # Rotate dates for readability
    plt.xticks(rotation=45)

    # Tight layout
    plt.tight_layout()

    # Show plot
    plt.show()

In [ ]:
plot_time_series(
    df1_imputed,
    date_col="date",
    columns=[
        # "complaints",
        # "media_mentions",
        # "backlog_days",
        # "staffing_level_fte",
        "centered_7d_mean"
    ]
)

## STATISTICAL ANALYSIS

## MODELLING AND/OR FORECASTING

In [ ]:
# import random
# import numpy as np
# import pandas as pd
# import holidays
# import matplotlib.pyplot as plt

# from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
# from sklearn.linear_model import Ridge
# from sklearn.multioutput import MultiOutputRegressor
# from sklearn.metrics import mean_absolute_percentage_error

# from skopt import BayesSearchCV
# from skopt.space import Integer, Real, Categorical

In [ ]:



SEED = 42
np.random.seed(SEED)


# -----------------------------
# Metrics
# -----------------------------
def mase(y_true, y_pred, y_train, seasonality=7):
    naive_error = np.mean(
        np.abs(y_train[seasonality:] - y_train[:-seasonality])
    )
    model_error = np.mean(np.abs(y_true - y_pred))
    return model_error / naive_error


def mape(y_true, y_pred):
    return mean_absolute_percentage_error(y_true, y_pred) * 100


# -----------------------------
# Date features + UK holidays
# -----------------------------
def add_date_features(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date")

    years = range(df["date"].dt.year.min(), df["date"].dt.year.max() + 1)
    uk_holidays = holidays.UK(years=years)

    df["is_weekend"] = df["date"].dt.dayofweek.isin([5, 6]).astype(int)
    df["bank_holiday_flag"] = df["date"].isin(uk_holidays).astype(int)

    df["dayofweek"] = df["date"].dt.dayofweek
    df["month"] = df["date"].dt.month
    df["dayofyear"] = df["date"].dt.dayofyear

    return df


# -----------------------------
# Lag features
# -----------------------------
def create_lag_features(df, target_col, lags=[1, 7, 14, 28]):
    df = df.copy()

    for lag in lags:
        df[f"{target_col}_lag_{lag}"] = df[target_col].shift(lag)

    df[f"{target_col}_rolling_7"] = (
        df[target_col].shift(1).rolling(7).mean()
    )

    return df


# -----------------------------
# Model factory
# -----------------------------
def get_model(trial, model_name):
    if model_name == "ridge":
        alpha = trial.suggest_float("alpha", 0.01, 100, log=True)
        return Ridge(alpha=alpha)

    if model_name == "random_forest":
        return RandomForestRegressor(
            n_estimators=trial.suggest_int("n_estimators", 100, 500),
            max_depth=trial.suggest_int("max_depth", 3, 20),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
            random_state=SEED,
            n_jobs=-1
        )

    if model_name == "hist_gradient_boosting":
        return HistGradientBoostingRegressor(
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3),
            max_iter=trial.suggest_int("max_iter", 100, 500),
            max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 10, 50),
            random_state=SEED
        )


# -----------------------------
# Bayesian tuning with Optuna
# -----------------------------
def tune_model(X, y, model_name, n_trials=30):
    tscv = TimeSeriesSplit(n_splits=3)

    def objective(trial):
        model = get_model(trial, model_name)
        scores = []

        for train_idx, valid_idx in tscv.split(X):
            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

            model.fit(X_train, y_train)
            preds = model.predict(X_valid)

            score = mape(y_valid, preds)
            scores.append(score)

        return np.mean(scores)

    sampler = optuna.samplers.TPESampler(seed=SEED)

    study = optuna.create_study(
        direction="minimize",
        sampler=sampler
    )

    study.optimize(objective, n_trials=n_trials)

    best_model = get_model(
        optuna.trial.FixedTrial(study.best_params),
        model_name
    )

    best_model.fit(X, y)

    return best_model, study.best_value, study.best_params


# -----------------------------
# Forecast one column recursively
# -----------------------------
def forecast_column(df, target_col, forecast_horizon=90):
    df = add_date_features(df)

    df_model = create_lag_features(df, target_col)
    df_model = df_model.dropna()

    feature_cols = [
        "is_weekend",
        "bank_holiday_flag",
        "dayofweek",
        "month",
        "dayofyear",
        f"{target_col}_lag_1",
        f"{target_col}_lag_7",
        f"{target_col}_lag_14",
        f"{target_col}_lag_28",
        f"{target_col}_rolling_7",
    ]

    X = df_model[feature_cols]
    y = df_model[target_col]

    models = [
        "ridge",
        "random_forest",
        "hist_gradient_boosting"
    ]

    best = None

    for model_name in models:
        model, score, params = tune_model(X, y, model_name)

        if best is None or score < best["score"]:
            best = {
                "model_name": model_name,
                "model": model,
                "score": score,
                "params": params
            }

    history = df[["date", target_col]].copy()

    future_dates = pd.date_range(
        start=df["date"].max() + pd.Timedelta(days=1),
        periods=forecast_horizon,
        freq="D"
    )

    forecasts = []

    for future_date in future_dates:
        temp = pd.DataFrame({
            "date": [future_date]
        })

        temp[target_col] = np.nan

        combined = pd.concat(
            [history, temp],
            ignore_index=True
        )

        combined = add_date_features(combined)
        combined = create_lag_features(combined, target_col)

        X_future = combined.iloc[[-1]][feature_cols]

        pred = best["model"].predict(X_future)[0]

        forecasts.append(pred)

        history = pd.concat(
            [
                history,
                pd.DataFrame({
                    "date": [future_date],
                    target_col: [pred]
                })
            ],
            ignore_index=True
        )

    forecast_df = pd.DataFrame({
        "date": future_dates,
        target_col: forecasts
    })

    return forecast_df, best


# -----------------------------
# Full multi-model forecasting pipeline
# -----------------------------
def forecast_recovered_complaints_90_days(df, horizon=90):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date")

    exog_cols = [
        "staffing_level_fte",
        "channel_mix_index",
        "backlog_days",
        "media_mentions"
    ]

    target_col = "recovered_complaints"

    future_df = pd.DataFrame({
        "date": pd.date_range(
            start=df["date"].max() + pd.Timedelta(days=1),
            periods=horizon,
            freq="D"
        )
    })

    future_df = add_date_features(future_df)

    model_info = {}

    # Forecast exogenous variables first
    for col in exog_cols:
        col_forecast, best_model = forecast_column(
            df[["date", col]].dropna(),
            target_col=col,
            forecast_horizon=horizon
        )

        future_df[col] = col_forecast[col].values
        model_info[col] = best_model

    # Now forecast recovered complaints
    df_target = df[
        [
            "date",
            target_col,
            "is_weekend",
            "bank_holiday_flag"
        ] + exog_cols
    ].copy()

    full_df = pd.concat(
        [
            df_target,
            future_df[
                [
                    "date",
                    "is_weekend",
                    "bank_holiday_flag"
                ] + exog_cols
            ]
        ],
        ignore_index=True
    )

    full_df = add_date_features(full_df)
    full_df = create_lag_features(full_df, target_col)

    feature_cols = [
        "is_weekend",
        "bank_holiday_flag",
        "dayofweek",
        "month",
        "dayofyear",
        "staffing_level_fte",
        "channel_mix_index",
        "backlog_days",
        "media_mentions",
        f"{target_col}_lag_1",
        f"{target_col}_lag_7",
        f"{target_col}_lag_14",
        f"{target_col}_lag_28",
        f"{target_col}_rolling_7",
    ]

    train_df = full_df[full_df[target_col].notna()].dropna()

    X = train_df[feature_cols]
    y = train_df[target_col]

    models = [
        "ridge",
        "random_forest",
        "hist_gradient_boosting"
    ]

    best = None

    for model_name in models:
        model, score, params = tune_model(X, y, model_name)

        if best is None or score < best["score"]:
            best = {
                "model_name": model_name,
                "model": model,
                "score": score,
                "params": params
            }

    history = df[[target_col]].copy()
    forecasts = []

    for i in range(horizon):
        row_idx = len(df) + i

        X_future = full_df.loc[[row_idx], feature_cols]

        pred = best["model"].predict(X_future)[0]

        full_df.loc[row_idx, target_col] = pred
        forecasts.append(pred)

        # update future lag features after prediction
        full_df = create_lag_features(full_df, target_col)

    future_df["forecast_recovered_complaints"] = forecasts

    model_info[target_col] = best

    return future_df, model_info

In [ ]:
forecast_90d, model_info = forecast_recovered_complaints_90_days(
    df1_imputed,
    horizon=90
)

print(forecast_90d.head())

In [ ]:
for col, info in model_info.items():
    print("\nColumn:", col)
    print("Best model:", info["model_name"])
    print("Best MAPE:", info["score"])
    print("Best params:", info["params"])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(
    df1_imputed["date"],
    df1_imputed["recovered_complaints"],
    label="Historical recovered complaints"
)

plt.plot(
    forecast_90d["date"],
    forecast_90d["forecast_recovered_complaints"],
    label="90-day forecast"
)

plt.xlabel("Date")
plt.ylabel("Recovered Complaints")
plt.title("90-Day Forecast of Recovered Complaints")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Count NaN values in each column
nan_counts = df1_imputed.isna().sum()

print(nan_counts)

In [ ]:
# pip install prophet pmdarima holidays

import numpy as np
import pandas as pd
import holidays

from prophet import Prophet
from pmdarima import auto_arima


def forecast_with_prophet_and_arima_exog(
    df,
    horizon=90,
    date_col="date",
    target_col="recovered_complaints",
    exog_cols=[
        "staffing_level_fte",
        "channel_mix_index",
        "backlog_days",
        "media_mentions"
    ],
    known_future_cols=[
        "is_weekend",
        "bank_holiday_flag"
    ],
    country_holidays="UK",
    seed=42
):

    np.random.seed(seed)

    df = df.copy()

    df[date_col] = pd.to_datetime(df[date_col])

    df = df.sort_values(date_col)

    # -----------------------------------
    # Forecast exogenous variables using ARIMA
    # -----------------------------------
    future_dates = pd.date_range(
        start=df[date_col].max() + pd.Timedelta(days=1),
        periods=horizon,
        freq="D"
    )

    future_exog = pd.DataFrame({
        "ds": future_dates
    })

    arima_models = {}

    for col in exog_cols:

        print(f"\nForecasting exogenous variable with ARIMA: {col}")

        series = df[col].dropna()

        model = auto_arima(
            series,
            seasonal=True,
            m=7,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore",
            random_state=seed
        )

        forecast_vals = model.predict(n_periods=horizon)

        future_exog[col] = forecast_vals
        print(forecast_vals)

        arima_models[col] = model

    # -----------------------------------
    # Known future variables
    # -----------------------------------
    if "is_weekend" in known_future_cols:

        future_exog["is_weekend"] = (
            future_exog["ds"]
            .dt.dayofweek
            .isin([5, 6])
            .astype(int)
        )

    if "bank_holiday_flag" in known_future_cols:

        years = range(
            future_exog["ds"].dt.year.min(),
            future_exog["ds"].dt.year.max() + 1
        )

        uk_holidays = holidays.UK(years=years)

        future_exog["bank_holiday_flag"] = (
            future_exog["ds"]
            .isin(uk_holidays)
            .astype(int)
        )

    # -----------------------------------
    # Prophet dataframe
    # -----------------------------------
    prophet_df = df[
        [date_col, target_col]
        + exog_cols
        + known_future_cols
    ].copy()

    prophet_df = prophet_df.rename(
        columns={
            date_col: "ds",
            target_col: "y"
        }
    )

    prophet_df = prophet_df.dropna(subset=["y"])

    # -----------------------------------
    # Prophet model
    # -----------------------------------
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False
    )

    model.add_country_holidays(
        country_name=country_holidays
    )

    # Add regressors
    for col in exog_cols + known_future_cols:
        model.add_regressor(col)

    # Fit
    model.fit(prophet_df)

    # -----------------------------------
    # Future dataframe
    # -----------------------------------
    future = model.make_future_dataframe(
        periods=horizon,
        freq="D"
    )

    # Historical regressors
    historical_regressors = df[
        [date_col]
        + exog_cols
        + known_future_cols
    ].rename(columns={date_col: "ds"})

    future = future.merge(
        historical_regressors,
        on="ds",
        how="left"
    )

    # Insert forecasted future exogenous values
    for col in exog_cols + known_future_cols:

        future.loc[
            future["ds"] > df[date_col].max(),
            col
        ] = future_exog[col].values
    future.to_csv("checking_prophet_future.csv")
    # -----------------------------------
    # Prophet forecast
    # -----------------------------------
    forecast = model.predict(future)

    forecast_90d = forecast[
        forecast["ds"] > df[date_col].max()
    ][
        [
            "ds",
            "yhat",
            "yhat_lower",
            "yhat_upper"
        ]
    ].copy()

    forecast_90d = forecast_90d.rename(
        columns={
            "ds": "date",
            "yhat": f"forecast_{target_col}",
            "yhat_lower": "lower_bound",
            "yhat_upper": "upper_bound"
        }
    )

    return (
        forecast_90d,
        forecast,
        model,
        arima_models,
        future_exog
    )

In [ ]:
forecast_90d, full_forecast, prophet_model, arima_models, future_exog = (
    forecast_with_prophet_and_arima_exog(
        df=df1_imputed,
        horizon=90
    )
)

print(forecast_90d.head())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(
    df1_imputed["date"],
    df1_imputed["recovered_complaints"],
    label="Historical"
)

plt.plot(
    forecast_90d["date"],
    forecast_90d["forecast_recovered_complaints"],
    label="Forecast"
)

plt.fill_between(
    forecast_90d["date"],
    forecast_90d["lower_bound"],
    forecast_90d["upper_bound"],
    alpha=0.2
)

plt.legend()

plt.xlabel("Date")

plt.ylabel("Recovered Complaints")

plt.title(
    "Prophet Forecast with ARIMA Forecasted Exogenous Variables"
)

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()

### PREPROCESSING

### MODEL TRAINING

In [ ]:
# pip install prophet holidays

import numpy as np
import pandas as pd
import holidays
from prophet import Prophet


def add_known_future_features(
    future_df,
    date_col="ds",
    country="UK"
):
    future_df = future_df.copy()

    # Weekend
    future_df["is_weekend"] = (
        future_df[date_col]
        .dt.dayofweek
        .isin([5, 6])
        .astype(int)
    )

    # UK bank holidays
    years = range(
        future_df[date_col].dt.year.min(),
        future_df[date_col].dt.year.max() + 1
    )

    uk_holidays = holidays.UK(years=years)

    future_df["bank_holiday_flag"] = (
        future_df[date_col]
        .isin(uk_holidays)
        .astype(int)
    )

    return future_df

In [ ]:
def forecast_single_series_with_prophet(
    df,
    target_col,
    horizon=90,
    date_col="date",
    country_holidays="UK",
    seed=42
):
    np.random.seed(seed)

    prophet_df = df[[date_col, target_col]].copy()
    prophet_df[date_col] = pd.to_datetime(prophet_df[date_col])

    prophet_df = prophet_df.rename(
        columns={
            date_col: "ds",
            target_col: "y"
        }
    )

    prophet_df = prophet_df.dropna(subset=["y"])

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False
    )

    model.add_country_holidays(country_name=country_holidays)

    model.fit(prophet_df)

    future = model.make_future_dataframe(
        periods=horizon,
        freq="D"
    )

    forecast = model.predict(future)

    future_forecast = forecast[
        forecast["ds"] > prophet_df["ds"].max()
    ][["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()

    future_forecast = future_forecast.rename(
        columns={
            "ds": "date",
            "yhat": target_col,
            "yhat_lower": f"{target_col}_lower",
            "yhat_upper": f"{target_col}_upper"
        }
    )

    return future_forecast, forecast, model

In [ ]:
# pip install prophet optuna holidays

import numpy as np
import pandas as pd
import optuna

from prophet import Prophet
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_percentage_error


def mase(y_true, y_pred, y_train, seasonality=7):

    naive_error = np.mean(
        np.abs(
            y_train[seasonality:] - y_train[:-seasonality]
        )
    )

    model_error = np.mean(
        np.abs(y_true - y_pred)
    )

    return model_error / naive_error


def forecast_single_series_with_prophet(
    df,
    target_col,
    horizon=90,
    date_col="date",
    country_holidays="UK",
    seed=42,
    n_trials=30,
    n_splits=3
):

    np.random.seed(seed)

    # -----------------------------------
    # Prepare dataframe
    # -----------------------------------
    prophet_df = df[[date_col, target_col]].copy()

    prophet_df[date_col] = pd.to_datetime(
        prophet_df[date_col]
    )

    prophet_df = prophet_df.rename(
        columns={
            date_col: "ds",
            target_col: "y"
        }
    )

    prophet_df = prophet_df.dropna(subset=["y"])

    prophet_df = prophet_df.sort_values("ds")

    # -----------------------------------
    # Time series CV
    # -----------------------------------
    tscv = TimeSeriesSplit(
        n_splits=n_splits
    )

    # -----------------------------------
    # Optuna objective
    # -----------------------------------
    def objective(trial):

        params = {

            "changepoint_prior_scale":
                trial.suggest_float(
                    "changepoint_prior_scale",
                    0.001,
                    0.5,
                    log=True
                ),

            "seasonality_prior_scale":
                trial.suggest_float(
                    "seasonality_prior_scale",
                    0.01,
                    20,
                    log=True
                ),

            "holidays_prior_scale":
                trial.suggest_float(
                    "holidays_prior_scale",
                    0.01,
                    20,
                    log=True
                ),

            "seasonality_mode":
                trial.suggest_categorical(
                    "seasonality_mode",
                    ["additive", "multiplicative"]
                )
        }

        mase_scores = []

        for train_idx, valid_idx in tscv.split(prophet_df):

            train_df = prophet_df.iloc[train_idx]
            valid_df = prophet_df.iloc[valid_idx]

            model = Prophet(
                yearly_seasonality=True,
                weekly_seasonality=True,
                daily_seasonality=False,
                **params
            )

            model.add_country_holidays(
                country_name=country_holidays
            )

            model.fit(train_df)

            future = valid_df[["ds"]]

            preds = model.predict(future)

            score = mase(
                y_true=valid_df["y"].values,
                y_pred=preds["yhat"].values,
                y_train=train_df["y"].values
            )

            mase_scores.append(score)

        return np.mean(mase_scores)

    # -----------------------------------
    # Bayesian optimization
    # -----------------------------------
    sampler = optuna.samplers.TPESampler(
        seed=seed
    )

    study = optuna.create_study(
        direction="minimize",
        sampler=sampler
    )

    study.optimize(
        objective,
        n_trials=n_trials,
        n_jobs=-1
    )

    best_params = study.best_params

    print("\nBest Prophet Params:")
    print(best_params)

    print("\nBest CV MASE:")
    print(study.best_value)

    # -----------------------------------
    # Train final model on full data
    # -----------------------------------
    final_model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        **best_params
    )

    final_model.add_country_holidays(
        country_name=country_holidays
    )

    final_model.fit(prophet_df)

    # -----------------------------------
    # Future forecast
    # -----------------------------------
    future = final_model.make_future_dataframe(
        periods=horizon,
        freq="D"
    )

    forecast = final_model.predict(future)

    future_forecast = forecast[
        forecast["ds"] > prophet_df["ds"].max()
    ][
        [
            "ds",
            "yhat",
            "yhat_lower",
            "yhat_upper"
        ]
    ].copy()

    future_forecast = future_forecast.rename(
        columns={
            "ds": "date",
            "yhat": target_col,
            "yhat_lower": f"{target_col}_lower",
            "yhat_upper": f"{target_col}_upper"
        }
    )

    # -----------------------------------
    # Final in-sample metrics
    # -----------------------------------
    full_preds = final_model.predict(
        prophet_df[["ds"]]
    )

    final_mape = mean_absolute_percentage_error(
        prophet_df["y"],
        full_preds["yhat"]
    )

    final_mase = mase(
        y_true=prophet_df["y"].values,
        y_pred=full_preds["yhat"].values,
        y_train=prophet_df["y"].values
    )

    print("\nFinal Metrics")
    print("MAPE:", round(final_mape, 4))
    print("MASE:", round(final_mase, 4))

    return (
        future_forecast,
        forecast,
        final_model,
        #study
    )

In [ ]:
def forecast_with_prophet_exog_using_prophet(
    df,
    horizon=90,
    date_col="date",
    target_col="recovered_complaints",
    exog_cols=[
        "staffing_level_fte",
        "channel_mix_index",
        "backlog_days",
        "media_mentions"
    ],
    known_future_cols=[
        "is_weekend",
        "bank_holiday_flag"
    ],
    country_holidays="UK",
    seed=42
):
    np.random.seed(seed)

    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)

    # -----------------------------------
    # 1. Forecast each unknown exogenous variable using Prophet
    # -----------------------------------
    future_exog = pd.DataFrame({
        "date": pd.date_range(
            start=df[date_col].max() + pd.Timedelta(days=1),
            periods=horizon,
            freq="D"
        )
    })

    exog_models = {}
    exog_full_forecasts = {}

    for col in exog_cols:

        print(f"\nForecasting exogenous variable with Prophet: {col}")

        col_future, col_full_forecast, col_model = (
            forecast_single_series_with_prophet(
                df=df,
                target_col=col,
                horizon=horizon,
                date_col=date_col,
                country_holidays=country_holidays,
                seed=seed
            )
        )

        future_exog[col] = col_future[col].values

        exog_models[col] = col_model
        exog_full_forecasts[col] = col_full_forecast

    # -----------------------------------
    # 2. Add known future features
    # -----------------------------------
    future_exog = future_exog.rename(columns={"date": "ds"})

    future_exog = add_known_future_features(
        future_exog,
        date_col="ds",
        country=country_holidays
    )

    future_exog = future_exog.rename(columns={"ds": "date"})

    # -----------------------------------
    # 3. Prepare target Prophet model
    # -----------------------------------
    prophet_df = df[
        [date_col, target_col]
        + exog_cols
        + known_future_cols
    ].copy()

    prophet_df = prophet_df.rename(
        columns={
            date_col: "ds",
            target_col: "y"
        }
    )

    prophet_df = prophet_df.dropna(subset=["y"])

    target_model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False
    )

    target_model.add_country_holidays(
        country_name=country_holidays
    )

    for col in exog_cols + known_future_cols:
        target_model.add_regressor(col)

    target_model.fit(prophet_df)

    # -----------------------------------
    # 4. Build future dataframe for target forecast
    # -----------------------------------
    future = target_model.make_future_dataframe(
        periods=horizon,
        freq="D"
    )

    historical_regressors = df[
        [date_col]
        + exog_cols
        + known_future_cols
    ].rename(columns={date_col: "ds"})

    future = future.merge(
        historical_regressors,
        on="ds",
        how="left"
    )

    future_exog_for_merge = future_exog.rename(
        columns={"date": "ds"}
    )

    future = future.merge(
        future_exog_for_merge,
        on="ds",
        how="left",
        suffixes=("", "_future")
    )

    for col in exog_cols + known_future_cols:
        future[col] = future[col].combine_first(
            future[f"{col}_future"]
        )

        if f"{col}_future" in future.columns:
            future = future.drop(columns=[f"{col}_future"])

    # -----------------------------------
    # 5. Forecast target
    # -----------------------------------
    forecast = target_model.predict(future)

    forecast_90d = forecast[
        forecast["ds"] > df[date_col].max()
    ][
        [
            "ds",
            "yhat",
            "yhat_lower",
            "yhat_upper"
        ]
    ].copy()

    forecast_90d = forecast_90d.rename(
        columns={
            "ds": "date",
            "yhat": f"forecast_{target_col}",
            "yhat_lower": "lower_bound",
            "yhat_upper": "upper_bound"
        }
    )

    # Include forecasted exogenous variables
    forecast_90d = forecast_90d.merge(
        future_exog,
        on="date",
        how="left"
    )

    return {
        "forecast_90d": forecast_90d,
        "target_full_forecast": forecast,
        "target_model": target_model,
        "exog_models": exog_models,
        "exog_full_forecasts": exog_full_forecasts,
        "future_exog": future_exog
    }

In [ ]:
results = forecast_with_prophet_exog_using_prophet(
    df=df1_imputed,
    horizon=90,
    target_col="recovered_complaints"
)

forecast_90d = results["forecast_90d"]

print(forecast_90d.head())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(
    df1_imputed["date"],
    df1_imputed["recovered_complaints"],
    label="Historical recovered complaints"
)

plt.plot(
    forecast_90d["date"],
    forecast_90d["forecast_recovered_complaints"],
    label="Prophet forecast"
)

plt.fill_between(
    forecast_90d["date"],
    forecast_90d["lower_bound"],
    forecast_90d["upper_bound"],
    alpha=0.2,
    label="Uncertainty interval"
)

plt.xlabel("Date")
plt.ylabel("Recovered Complaints")
plt.title("90-Day Prophet Forecast with Prophet-Forecasted Exogenous Variables")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### PERFORMANCE EVALUATION

### PREDICTION

### SAVE THE MODEL